# Prezentacja Projektu ETL: Od danych surowych do analityki biznesowej
Ten notatnik jest główną osią demonstracji hurtowni danych. Pokazuje on pełen cykl życia danych:
1. **Staging (Strefa lądowania)** - Surowe, nieustrukturyzowane i "brudne" dane.
2. **Data Warehouse (Model Gwiazdy)** - Wyczyszczone wymiary i zdenormalizowane tabele faktów.
3. **Warstwa Semantyczna (Semantic Layer)** - Kręgosłup analityczny.


In [1]:
import os
import sys
# Dodajemy ścieżkę projektu do sys.path, aby móc importować moduły z src
sys.path.append(os.path.abspath('..'))
import pandas as pd
from src.utils.db import sqlserver_connection
# Ustawienia wyświetlania dla pandas (aby pokazać wszystkie kolumny)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
def query_db(sql_query: str) -> pd.DataFrame:
    with sqlserver_connection() as conn:
        return pd.read_sql(sql_query, conn)
print("Zależności i połączenie z bazą gotowe!")


Zależności i połączenie z bazą gotowe!


## Krok 1: Dane Surowe (Staging) - "Przed"
W tabeli `stg.iowa_liquor_sales_raw` dane lądują dokładnie tak, jak przyszły z plików płaskich CSV (lub API). Z biznesowego punktu widzenia takie dane są bezużyteczne i podatne na błędy.
**Dlaczego takie transformacje były konieczne?**
- Finanse mają znak dolara `$` i typ tekstowy (np. `$14.50`). Nie da się ich zsumować w SQL.
- Lokalizacje są zapisane jako łańcuch `POINT (-91.123 41.345)`. Nie zrobimy z tego mapy bez parsowania.
- Wiele kategorii (`category_name`) to zwyczajne braki danych. Tabela faktów nie może mieć pustych kluczy obcych (zgodnie z dobrymi praktykami modelu Kimballa).


In [2]:
sql = """
SELECT TOP 5 
    invoice_and_item_number,
    date,
    store_location,      -- Format przestrzenny POINT
    category_name,       -- Brakujące dane
    state_bottle_cost,   -- Znak $ i tekst
    sale_dollars         -- Znak $ i tekst
FROM stg.iowa_liquor_sales_raw;
"""
display(query_db(sql))


/tmp/ipykernel_933/4277566135.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,invoice_and_item_number,date,store_location,category_name,state_bottle_cost,sale_dollars
0,INV-54554000001,2023-01-02,POINT (-93.61378 41.60575),100% AGAVE TEQUILA,14.50,261.00
1,INV-54554000002,2023-01-02,POINT (-93.61378 41.60575),AMERICAN VODKAS,4.65,418.80
2,INV-54554000003,2023-01-02,POINT (-93.61378 41.60575),IMPORTED FLAVORED VODKA,9.96,358.56
3,INV-54554000004,2023-01-02,POINT (-93.61378 41.60575),CREAM LIQUEURS,17.00,306.00
4,INV-54554000005,2023-01-02,POINT (-93.61378 41.60575),SPICED RUM,12.49,1124.40


## Krok 2: Oczyszczone wymiary i fakty (Data Warehouse) - "Po"
Po uruchomieniu pipeline'u ETL opartego na Pythonie (Pandas) i zrzucie paczkami do bazy (Fast Executemany), otrzymujemy model Gwiazdy. 
- Wyłuskaliśmy matematyczne współrzędne z `POINT` na natywne `FLOAT` (latitude, longitude).
- Zabezpieczyliśmy braki kategorii poprzez sztuczny systemowy rekord `UNKNOWN`.
- Czyste finanse (DECIMAL) pozwalają na wyliczenie miar pochodnych już w tabeli faktów (np. `margin_amount`).


In [3]:
print("1. Wymiar Sklepu (Wyłuskane współrzędne):")
display(query_db("SELECT TOP 5 store_key, store_name, latitude, longitude FROM dw.dim_store;"))
print("\n2. Tabela Faktów (Czyste finanse i wyliczona marża):")
display(query_db("SELECT TOP 5 invoice_number, sale_dollars, state_bottle_cost, margin_amount FROM dw.fact_sales;"))
print("\n3. Rekord zabezpieczający braki danych w wymiarze Kategorii:")
display(query_db("SELECT * FROM dw.dim_category WHERE category_number = 'UNKNOWN';"))


1. Wymiar Sklepu (Wyłuskane współrzędne):


/tmp/ipykernel_933/4277566135.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,store_key,store_name,latitude,longitude
0,1,JACK & JILL STORE / WEST BRANCH,41.670494,-91.343413
1,2,LOCAL LIQUOR / PANORA,41.692594,-94.357208
2,3,LEGENDARY RYE / BAD BEAR ENTERPRISES (ET),42.067186,-94.866874
3,4,KWIK STAR #1158 / AMES,42.008984,-93.587699
4,5,EMPIRE LIQUOR AND TOBACCO / HIAWATHA,42.046042,-91.673047



2. Tabela Faktów (Czyste finanse i wyliczona marża):


/tmp/ipykernel_933/4277566135.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,invoice_number,sale_dollars,state_bottle_cost,margin_amount
0,INV-54554000001,261.00,14.50,87.00
1,INV-54554000002,418.80,4.65,139.80
2,INV-54554000003,358.56,9.96,119.52
3,INV-54554000004,306.00,17.00,102.00
4,INV-54554000005,1124.40,12.49,375.00



3. Rekord zabezpieczający braki danych w wymiarze Kategorii:


/tmp/ipykernel_933/4277566135.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,category_key,category_number,category_name


## Krok 3: Warstwa Semantyczna (Semantic Layer) - Kręgosłup biznesowy
**Dlaczego mamy aż 16 widoków?**
Hurtownia danych powinna służyć różnym celom i odpowiadać na pytania zadawane przez rożne działy w firmie (marketing, logistyka, zarząd). Warstwa semantyczna agreguje zdenormalizowany model gwiazdy w gotowe odpowiedzi. Zbudowaliśmy 16 widoków, aby rozdzielić te domeny logiczne:
1. **Analiza czasu**: `vw_sales_overview`, `vw_sales_by_day_type`, `vw_sales_by_month`, `vw_avg_sales_per_store_by_month_region`.
2. **Analiza asortymentu**: `vw_category_sales_over_time`, `vw_sales_by_category`, `vw_top_products`, `vw_margin_analysis`, `vw_sales_by_packaging`, `vw_sales_by_vendor`.
3. **Analiza geograficzna**: `vw_sales_by_geography`, `vw_sales_map_points`, `vw_sales_by_store`, `vw_volume_vs_revenue`.
4. **Wskaźniki ogólne**: `vw_kpi_summary`, `vw_etl_status`.
Dzięki nim aplikacje docelowe (takie jak nasz panel w Streamlit) wykonują proste `SELECT * FROM widok` i nie obciążają bazy ciężkimi złączeniami ani funkcjami agregującymi.
---
### Analiza 7 Najważniejszych Widoków (Top 7)
Przyjrzyjmy się głębiej tym siedmiu widokom, które są esencją i fundamentem naszego rozwiązania.


### 1. `sem.vw_kpi_summary` (Wysokopoziomowy pogląd na biznes)
**Dlaczego ten widok?**
Zawiera kluczowe wskaźniki efektywności liczone dla całej firmy. Dział zarządu nie musi analizować setek tysięcy paragonów – potrzebują wiedzieć, jaka jest średnia wielkość koszyka (`avg_invoice_value`) i procentowa marżowość (`avg_margin_percent`).


In [4]:
display(query_db("SELECT * FROM sem.vw_kpi_summary;"))


/tmp/ipykernel_933/4277566135.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,total_sales,total_margin,sales_line_count,total_bottles_sold,total_volume_liters,invoice_count,store_count,product_count,category_count,vendor_count,avg_invoice_value,avg_bottles_per_invoice,avg_margin_percent,sales_per_store,sales_per_liter
0,4.466417e+08,1.492504e+08,2635879,31302201.0,23756277.41,2635879,2110,5231,48,250,169.446953,11.875431,33.42,211678.514578,18.800995


### 2. `sem.vw_etl_status` (Zdrowie potoku danych)
**Dlaczego ten widok?**
Widok ten pozwala na monitoring i audyt ETL w czasie rzeczywistym. Pokazuje zliczenia wierszy we wszystkich wymiarach, by łatwo wykryć, czy dzisiejszy przyrost danych załadował się pomyślnie.


In [5]:
display(query_db("SELECT * FROM sem.vw_etl_status;"))


/tmp/ipykernel_933/4277566135.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,status_generated_at,staging_row_count,fact_row_count,dim_date_count,dim_store_count,dim_product_count,dim_category_count,dim_vendor_count,dim_packaging_count,min_date,max_date,last_staging_load_timestamp,last_fact_load_timestamp
0,2026-07-04 21:18:52.450,2639557,2635879,290,2110,5231,48,250,69,2023-01-02,2023-12-30,2026-07-04 19:53:26.060703,2026-07-04 19:53:43.247454


### 3. `sem.vw_sales_by_month` (Klasyczna analiza szeregów czasowych)
**Dlaczego ten widok?**
Odpowiada na fundamentalne pytanie o trend przychodów. Grupuje setki tysięcy faktów w pojedyncze miesiące. Dzięki niemu analityk w sekundę wyrysuje wykres przychodów na osi czasu.


In [6]:
display(query_db("SELECT TOP 5 * FROM sem.vw_sales_by_month ORDER BY year, month;"))


/tmp/ipykernel_933/4277566135.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,year,quarter,month,year_month,total_sales,total_bottles_sold,total_volume_liters,total_margin,sales_line_count,invoice_count,store_count
0,2023,1,1,2023-01,32582340.63,2358449.0,1747102.70,10888433.83,212850,212850,1857
1,2023,1,2,2023-02,32134462.65,2289374.0,1771119.28,10750429.35,189294,189294,1824
2,2023,1,3,2023-03,36436060.72,2632521.0,2016226.87,12184903.75,221313,221313,1859
3,2023,2,4,2023-04,32915910.22,2393665.0,1793682.48,10986201.79,198446,198446,1832
4,2023,2,5,2023-05,39721449.29,2786013.0,2187271.06,13398186.64,233060,233060,1879


### 4. `sem.vw_sales_by_day_type` (Zachowania w Weekend vs Dzień Roboczy)
**Dlaczego ten widok?**
Prezentuje on możliwości wymiaru `dim_date`, w którym zaszyta jest reguła biznesowa w postaci flagi `is_weekend`. Baza na poziomie silnika potrafi dzięki temu oddzielić analizę zysku dni roboczych od wolnych, bez potrzeby pisania logiki obliczania dni tygodnia w kodzie aplikacji.


In [7]:
display(query_db("SELECT * FROM sem.vw_sales_by_day_type;"))


/tmp/ipykernel_933/4277566135.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,year,quarter,month,year_month,is_weekend,day_type,total_sales,total_bottles_sold,total_volume_liters,total_margin,sales_line_count,invoice_count
0,2023,3,7,2023-07,True,Weekend,1186208.53,87163.0,61733.02,395977.14,8854,8854
1,2023,1,1,2023-01,True,Weekend,838436.05,66807.0,45927.71,279843.16,7061,7061
2,2023,3,9,2023-09,True,Weekend,1728305.20,128850.0,95695.29,577184.04,12263,12263
3,2023,2,6,2023-06,True,Weekend,1424375.15,105970.0,76931.37,474511.15,9701,9701
4,2023,3,8,2023-08,True,Weekend,298.45,12.0,15.00,102.50,7,7
5,2023,4,11,2023-11,True,Weekend,3412682.64,271801.0,178442.29,1138208.97,20659,20659
6,2023,1,3,2023-03,False,Weekday,36433363.96,2632359.0,2016057.37,12183998.11,221299,221299
7,2023,4,11,2023-11,False,Weekday,35929764.34,2395754.0,1884156.66,11984428.10,202653,202653
8,2023,4,12,2023-12,True,Weekend,3407799.86,262935.0,170366.75,1136876.98,24738,24738
9,2023,2,5,2023-05,False,Weekday,39704733.36,2784835.0,2186163.82,13392595.99,232960,232960


### 5. `sem.vw_sales_by_category` (Funkcje Okna i udziały rynkowe)
**Dlaczego ten widok?**
To przykład zaawansowanego SQL. Użyliśmy złączeń krzyżowych (`CROSS JOIN`) z pre-kalkulowaną pulą całkowitej sprzedaży (`totals`), by dla każdej kategorii alkoholi wyliczyć jej procentowy udział w całkowitych przychodach (`sales_share_percent`). Liczenie tego w Tableau lub Pythonie byłoby mniej wydajne i podatne na błędy analityków.


In [8]:
display(query_db("SELECT TOP 5 * FROM sem.vw_sales_by_category ORDER BY total_sales DESC;"))


/tmp/ipykernel_933/4277566135.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,category_name,total_sales,total_bottles_sold,total_volume_liters,total_margin,sales_line_count,avg_margin_per_bottle,sales_share_percent
0,AMERICAN VODKAS,67412480.47,6513606.0,5796618.59,22496066.21,409705,3.453703,15.09
1,CANADIAN WHISKIES,50411669.32,3063909.0,2951534.33,16848279.17,252193,5.498948,11.29
2,STRAIGHT BOURBON WHISKIES,38747595.32,1715971.0,1360371.95,12932055.73,200909,7.536290,8.68
3,100% AGAVE TEQUILA,32384034.97,1109830.0,767620.07,10812614.24,128195,9.742586,7.25
4,WHISKEY LIQUEUR,26564898.28,4424696.0,1100176.85,8869781.44,165922,2.004608,5.95


### 6. `sem.vw_margin_analysis` (Czysta opłacalność)
**Dlaczego ten widok?**
Wyprowadza on wnioski bezpośrednio z faktu, że oczyściliśmy ceny z symboli `$`. Wyliczamy tutaj średnią różnicę pomiędzy ceną hurtową a detaliczną na poziomie jednostki (butelki), by wyłonić produkty o najwyższej stopie zwrotu (ROI).


In [9]:
display(query_db("SELECT TOP 5 category_name, vendor_name, avg_unit_margin, total_margin FROM sem.vw_margin_analysis ORDER BY total_margin DESC;"))


/tmp/ipykernel_933/4277566135.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,category_name,vendor_name,avg_unit_margin,total_margin
0,AMERICAN VODKAS,FIFTH GENERATION INC,5.734500,10027926.96
1,CANADIAN WHISKIES,HEAVEN HILL BRANDS,3.629779,4119795.00
2,WHISKEY LIQUEUR,SAZERAC COMPANY INC,2.815516,3879317.94
3,CANADIAN WHISKIES,DIAGEO AMERICAS,8.952332,3410927.00
4,TENNESSEE WHISKIES,BROWN FORMAN CORP.,9.499180,3211062.02


### 7. `sem.vw_sales_map_points` (Gotowe dane geolokalizacyjne)
**Dlaczego ten widok?**
To specjalnie spreparowany widok na potrzeby rysowania map w narzędziach BI (Streamlit). Posiada on wewnętrzną klauzulę filtrującą `WHERE latitude IS NOT NULL`, gwarantując że do komponentu renderującego mapę nigdy nie trafi "zepsuty" koordynat. To świetny przykład przygotowania danych skrojonych idealnie pod konkretny front-end aplikacji.


In [10]:
display(query_db("SELECT TOP 5 store_name, city, latitude, longitude, total_sales FROM sem.vw_sales_map_points ORDER BY total_sales DESC;"))


/tmp/ipykernel_933/4277566135.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,store_name,city,latitude,longitude,total_sales
0,HY-VEE #3 / BDI / DES MOINES,Des Moines,41.554269,-93.594781,15275187.46
1,CENTRAL CITY 2,Des Moines,41.605835,-93.613286,13740361.39
2,ANOTHER ROUND / DEWITT,Dewitt,41.809631,-90.538996,6894898.17
3,HY-VEE WINE AND SPIRITS #1 (1281) / IOWA CITY,Iowa City,41.642516,-91.529426,6112597.82
4,BENZ DISTRIBUTING,Cedar Rapids,41.975513,-91.659640,5235975.06
